# Credit-Conditioned Factor Rotation Strategy

This notebook contains the full core-strategy code directly inside the notebook. Each major step is separated into its own section, with a short explanation followed by the code that implements it.


## 1. Imports And Project Configuration

This section imports the libraries used throughout the notebook and defines the file paths, sample window, and fixed regime weights from the strategy context.


In [ ]:
from dataclasses import dataclass
from io import StringIO
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pandas.tseries.offsets import MonthEnd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'Data'
SAMPLE_START = pd.Timestamp('1990-01-31')
SAMPLE_END = pd.Timestamp('2024-09-30')
OUTPUT_CSV = PROJECT_ROOT / 'credit_factor_rotation_returns.csv'
OUTPUT_PLOT = PROJECT_ROOT / 'credit_factor_rotation_regime_returns.png'

BAD_REGIME_WEIGHTS = {'w_LV': 0.40, 'w_P': 0.30, 'w_I': 0.30}
GOOD_REGIME_WEIGHTS = {'w_LV': -0.40, 'w_P': 0.30, 'w_I': -0.30}


## 2. Helper Objects And Utility Functions

This section defines the container that stores intermediate outputs and the small helper functions used repeatedly in portfolio construction, date alignment, and value-weighted return aggregation.


In [ ]:
@dataclass
class StrategyArtifacts:
    crsp: pd.DataFrame
    fundamentals: pd.DataFrame
    linked_signals: pd.DataFrame
    factor_returns: pd.DataFrame
    factor_validation: pd.DataFrame
    macro_signal: pd.DataFrame
    strategy_panel: pd.DataFrame
    strategy_returns: pd.DataFrame


def to_month_end(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, errors='coerce') + MonthEnd(0)


def weighted_quintile_returns(panel: pd.DataFrame) -> pd.DataFrame:
    data = panel.copy()
    valid = data['ret'].notna()
    data['weight_valid'] = np.where(valid, data['weight'], 0.0)
    data['weighted_ret'] = np.where(valid, data['ret'] * data['weight'], 0.0)

    grouped = (
        data.groupby(['date', 'quintile'], sort=True)[['weight_valid', 'weighted_ret']]
        .sum()
        .reset_index()
    )
    grouped['vwret'] = grouped['weighted_ret'] / grouped['weight_valid']

    pivot = grouped.pivot(index='date', columns='quintile', values='vwret').sort_index()
    pivot.columns = [f'Q{int(col)}' for col in pivot.columns]
    return pivot


def assign_quintiles(universe: pd.DataFrame, signal_col: str, date_col: str) -> pd.DataFrame:
    breakpoints = (
        universe.loc[universe['exchcd'] == 1]
        .groupby(date_col)[signal_col]
        .quantile([0.2, 0.4, 0.6, 0.8])
        .unstack()
        .rename(columns={0.2: 'q20', 0.4: 'q40', 0.6: 'q60', 0.8: 'q80'})
    )

    ranked = universe.merge(
        breakpoints,
        left_on=date_col,
        right_index=True,
        how='inner',
    )

    ranked['quintile'] = np.select(
        [
            ranked[signal_col] <= ranked['q20'],
            ranked[signal_col] <= ranked['q40'],
            ranked[signal_col] <= ranked['q60'],
            ranked[signal_col] <= ranked['q80'],
            ranked[signal_col] > ranked['q80'],
        ],
        [1, 2, 3, 4, 5],
        default=np.nan,
    )

    ranked = ranked.dropna(subset=['quintile']).copy()
    ranked['quintile'] = ranked['quintile'].astype('int8')
    return ranked


## 3. Load The CRSP Monthly Stock File

This section reads the monthly stock panel, converts dates to month-end, coerces CRSP fields safely to numeric, filters the sample to the required common stocks and exchanges, and creates the two rolling inputs used later: the 24-month history flag and the 12-month low-volatility signal.


In [ ]:
def load_crsp(path: Path | None = None) -> pd.DataFrame:
    path = path or DATA_DIR / 'crsp_clean_filtered(in).csv'
    crsp = pd.read_csv(
        path,
        usecols=['permno', 'date', 'shrcd', 'exchcd', 'siccd', 'ret', 'me'],
        dtype={
            'permno': 'int32',
            'shrcd': 'string',
            'exchcd': 'string',
            'siccd': 'string',
            'ret': 'string',
            'me': 'string',
        },
        low_memory=False,
    )

    crsp['date'] = to_month_end(crsp['date'])
    for col in ['shrcd', 'exchcd', 'siccd', 'ret', 'me']:
        crsp[col] = pd.to_numeric(crsp[col], errors='coerce')

    crsp = crsp.loc[
        crsp['date'].between(SAMPLE_START, pd.Timestamp('2024-12-31'))
        & crsp['shrcd'].isin([10, 11])
        & crsp['exchcd'].isin([1, 3])
    ].copy()

    crsp = crsp.sort_values(['permno', 'date']).reset_index(drop=True)

    valid_ret = crsp['ret'].notna().astype('int8')
    crsp['ret_history_24'] = (
        valid_ret.groupby(crsp['permno'])
        .rolling(24, min_periods=24)
        .sum()
        .reset_index(level=0, drop=True)
        .eq(24)
    )
    crsp['lowvol_signal'] = (
        crsp.groupby('permno')['ret']
        .rolling(12, min_periods=12)
        .std(ddof=1)
        .reset_index(level=0, drop=True)
    )

    return crsp


crsp = load_crsp()
crsp[['permno', 'date', 'ret', 'me', 'ret_history_24', 'lowvol_signal']].head()


## 4. Load Fundamentals And Compute Accounting Signals

This section loads the annual fundamentals file, applies the Compustat-style filters from the strategy note, uses `sale` as the revenue proxy, and computes gross profitability to assets and asset growth. Because this extract does not include a filing-date field, the code treats `datadate` as the earliest observable availability proxy and delays portfolio entry until the first matched CRSP month-end after that date.


In [ ]:
def load_fundamentals(path: Path | None = None) -> pd.DataFrame:
    path = path or DATA_DIR / 'QMJ_data.csv'
    funda = pd.read_csv(
        path,
        usecols=['gvkey', 'datadate', 'fyear', 'sale', 'cogs', 'at', 'indfmt', 'datafmt', 'consol'],
        dtype={'gvkey': 'string'},
        low_memory=False,
    )

    funda['datadate'] = pd.to_datetime(funda['datadate'], errors='coerce')
    for col in ['sale', 'cogs', 'at']:
        funda[col] = pd.to_numeric(funda[col], errors='coerce')

    funda = funda.loc[
        (funda['indfmt'] == 'INDL')
        & (funda['datafmt'] == 'STD')
        & (funda['consol'] == 'C')
        & funda['datadate'].notna()
    ].copy()

    funda = funda.sort_values(['gvkey', 'datadate']).reset_index(drop=True)
    funda['gpoa'] = (funda['sale'] - funda['cogs']) / funda['at']
    funda['at_lag'] = funda.groupby('gvkey')['at'].shift(1)
    funda['inv'] = (funda['at'] - funda['at_lag']) / funda['at_lag']

    return funda[['gvkey', 'datadate', 'fyear', 'gpoa', 'inv']]


fundamentals = load_fundamentals()
fundamentals.head()


## 5. Load The CCM Link Table And Link Fundamentals To CRSP

This section loads the valid CCM links, keeps only the accounting observations whose `datadate` falls inside the valid link interval, and maps each signal to the first CRSP month-end for the same `permno` on or after `datadate`. Signals that cannot be matched within six months are dropped as stale.


In [ ]:
MAX_SIGNAL_FORMATION_LAG_MONTHS = 6


def load_ccm(path: Path | None = None) -> pd.DataFrame:
    path = path or DATA_DIR / 'crsp_a_ccm.csv'
    ccm = pd.read_csv(
        path,
        usecols=['gvkey', 'LINKPRIM', 'LINKTYPE', 'LPERMNO', 'LINKDT', 'LINKENDDT'],
        dtype={'gvkey': 'string', 'LINKPRIM': 'string', 'LINKTYPE': 'string', 'LPERMNO': 'Int64'},
        low_memory=False,
    )

    ccm = ccm.loc[
        ccm['LINKTYPE'].isin(['LU', 'LC']) & ccm['LINKPRIM'].isin(['P', 'C'])
    ].copy()

    ccm['LINKDT'] = pd.to_datetime(ccm['LINKDT'], errors='coerce', format='mixed', dayfirst=True)
    ccm['LINKENDDT'] = pd.to_datetime(
        ccm['LINKENDDT'].replace({'E': None}),
        errors='coerce',
        format='mixed',
        dayfirst=True,
    ).fillna(pd.Timestamp('2100-12-31'))

    ccm = ccm.rename(columns={'LINKDT': 'linkdt', 'LINKENDDT': 'linkenddt', 'LPERMNO': 'permno'})
    ccm['permno'] = ccm['permno'].astype('int32')
    return ccm[['gvkey', 'permno', 'linkdt', 'linkenddt']]


def link_fundamentals_to_crsp(
    funda: pd.DataFrame,
    ccm: pd.DataFrame,
    crsp: pd.DataFrame,
    max_months_after_availability: int = MAX_SIGNAL_FORMATION_LAG_MONTHS,
) -> pd.DataFrame:
    linked = funda.merge(ccm, on='gvkey', how='inner')
    linked = linked.loc[
        linked['datadate'].between(linked['linkdt'], linked['linkenddt'])
    ].copy()

    crsp_months = (
        crsp[['permno', 'date']]
        .drop_duplicates()
        .sort_values(['permno', 'date'])
        .groupby('permno')['date']
        .apply(lambda s: s.to_numpy(dtype='datetime64[ns]'))
        .to_dict()
    )

    matched_groups = []
    for permno, group in linked.sort_values(['permno', 'datadate']).groupby('permno', sort=False):
        group = group.copy()
        dates = crsp_months.get(int(permno))
        if dates is None or len(dates) == 0:
            group['formation_date'] = pd.NaT
        else:
            availability = group['datadate'].to_numpy(dtype='datetime64[ns]')
            match_idx = np.searchsorted(dates, availability, side='left')
            matched = np.full(len(group), np.datetime64('NaT'), dtype='datetime64[ns]')
            valid = match_idx < len(dates)
            matched[valid] = dates[match_idx[valid]]
            group['formation_date'] = pd.to_datetime(matched)
        matched_groups.append(group)

    linked = pd.concat(matched_groups, ignore_index=True)

    max_formation_date = linked['datadate'] + MonthEnd(max_months_after_availability)
    linked = linked.loc[
        linked['formation_date'].notna() & linked['formation_date'].le(max_formation_date)
    ].copy()

    linked = linked.sort_values(['permno', 'formation_date', 'datadate'])
    linked = linked.drop_duplicates(['permno', 'formation_date'], keep='last')
    return linked[['permno', 'formation_date', 'gpoa', 'inv']]


ccm = load_ccm()
linked_signals = link_fundamentals_to_crsp(fundamentals, ccm, crsp)
linked_signals.head()


## 6. Build The Rolling Profitability And Investment Factors

This section replaces the old June-only convention with a rolling monthly implementation. Each stock carries its latest matched accounting signal forward from the first eligible CRSP month-end, monthly sorts use NYSE breakpoints and value weights, and the resulting portfolios earn next-month returns after the 24-month history and non-financial screens are applied.


In [ ]:
def build_annual_factor_returns(
    crsp: pd.DataFrame,
    linked_signals: pd.DataFrame,
    signal_col: str,
    factor_name: str,
    long_high_signal: bool,
    exclude_financials: bool,
) -> pd.Series:
    signal_updates = linked_signals[['permno', 'formation_date', signal_col]].dropna().copy()
    signal_updates = signal_updates.sort_values(['permno', 'formation_date'])
    signal_updates = signal_updates.drop_duplicates(['permno', 'formation_date'], keep='last')
    signal_updates = signal_updates.rename(columns={'formation_date': 'signal_date'})

    monthly_universe = crsp.loc[:, ['permno', 'date', 'exchcd', 'siccd', 'me', 'ret_history_24']].copy()
    monthly_universe = monthly_universe.merge(
        signal_updates,
        left_on=['permno', 'date'],
        right_on=['permno', 'signal_date'],
        how='left',
    )
    monthly_universe[signal_col] = monthly_universe.groupby('permno')[signal_col].ffill()

    universe = monthly_universe.loc[
        monthly_universe['ret_history_24'] & monthly_universe['me'].gt(0) & monthly_universe[signal_col].notna()
    ].copy()

    if exclude_financials:
        universe = universe.loc[~universe['siccd'].between(6000, 6999, inclusive='both')].copy()

    universe = assign_quintiles(universe, signal_col=signal_col, date_col='date')
    universe['weight'] = universe['me'] / universe.groupby(['date', 'quintile'])['me'].transform('sum')
    universe = universe.rename(columns={'date': 'formation_date'})

    monthly = crsp.loc[:, ['permno', 'date', 'ret']].copy()
    monthly['formation_date'] = monthly['date'] - MonthEnd(1)

    panel = monthly.merge(
        universe[['permno', 'formation_date', 'quintile', 'weight']],
        on=['permno', 'formation_date'],
        how='inner',
    )

    quintile_returns = weighted_quintile_returns(panel)
    long_leg = 'Q5' if long_high_signal else 'Q1'
    short_leg = 'Q1' if long_high_signal else 'Q5'
    return (quintile_returns[long_leg] - quintile_returns[short_leg]).rename(factor_name)


R_prof = build_annual_factor_returns(
    crsp=crsp,
    linked_signals=linked_signals,
    signal_col='gpoa',
    factor_name='R_prof',
    long_high_signal=True,
    exclude_financials=True,
)

R_inv = build_annual_factor_returns(
    crsp=crsp,
    linked_signals=linked_signals,
    signal_col='inv',
    factor_name='R_inv',
    long_high_signal=False,
    exclude_financials=True,
)

pd.concat([R_prof.rename('profitability'), R_inv.rename('investment')], axis=1).dropna().head()


## 7. Build The Monthly Low-Volatility Factor

This section forms the low-volatility factor each month using trailing 12-month return volatility, NYSE quintile breakpoints, and value weights. The monthly sort is then matched to next-month returns to create the factor return series.


In [ ]:
def build_lowvol_factor_returns(crsp: pd.DataFrame) -> pd.Series:
    universe = crsp.loc[
        crsp['ret_history_24'] & crsp['lowvol_signal'].notna() & crsp['me'].gt(0),
        ['permno', 'date', 'exchcd', 'me', 'lowvol_signal'],
    ].copy()
    universe = universe.rename(columns={'date': 'formation_date'})
    universe = assign_quintiles(universe, signal_col='lowvol_signal', date_col='formation_date')
    universe['weight'] = universe['me'] / universe.groupby(['formation_date', 'quintile'])['me'].transform('sum')

    monthly = crsp.loc[:, ['permno', 'date', 'ret']].copy()
    monthly['formation_date'] = monthly['date'] - MonthEnd(1)

    panel = monthly.merge(
        universe[['permno', 'formation_date', 'quintile', 'weight']],
        on=['permno', 'formation_date'],
        how='inner',
    )

    quintile_returns = weighted_quintile_returns(panel)
    return (quintile_returns['Q1'] - quintile_returns['Q5']).rename('R_lowvol')


R_lowvol = build_lowvol_factor_returns(crsp)
R_lowvol.dropna().head()


## 8. Load Macro Data And Build The Regime Signal

This section loads the macro yield series, computes the TERM and DEF variables, applies expanding-window standardization, forms the composite credit score, applies the expanding median regime rule, and shifts the regime decision forward one month to create the holding-date signal.


In [ ]:
def load_reuters_monthly(path: Path, value_name: str) -> pd.DataFrame:
    data = pd.read_csv(path, skiprows=[1])
    data = data.rename(columns={data.columns[0]: 'raw_date', data.columns[1]: value_name})
    data['date'] = pd.to_datetime(data['raw_date'], dayfirst=True, errors='coerce') + MonthEnd(0)
    data[value_name] = pd.to_numeric(data[value_name], errors='coerce')
    data = data[['date', value_name]].dropna().drop_duplicates('date').sort_values('date')
    return data.reset_index(drop=True)


def load_fred_monthly(path: Path, value_name: str) -> pd.DataFrame:
    data = pd.read_csv(path)
    data = data.rename(columns={data.columns[0]: 'raw_date', data.columns[1]: value_name})
    data['date'] = pd.to_datetime(data['raw_date'], errors='coerce') + MonthEnd(0)
    data[value_name] = pd.to_numeric(data[value_name], errors='coerce')
    data = data[['date', value_name]].dropna().drop_duplicates('date').sort_values('date')
    return data.reset_index(drop=True)


def build_macro_signal() -> pd.DataFrame:
    y10 = load_reuters_monthly(DATA_DIR / '10Y monthly US(Table Data).csv', 'yield_10y')
    y3m = load_reuters_monthly(DATA_DIR / 'US 3M monthly(Table Data).csv', 'yield_3m')
    aaa = load_fred_monthly(DATA_DIR / 'DAAA.csv', 'yield_aaa')
    baa = load_fred_monthly(DATA_DIR / 'DBAA.csv', 'yield_baa')

    macro = y10.merge(y3m, on='date', how='inner').merge(aaa, on='date', how='inner').merge(baa, on='date', how='inner')
    macro = macro.sort_values('date').reset_index(drop=True)
    macro = macro.loc[macro['date'].between(SAMPLE_START, SAMPLE_END)].copy()

    macro['TERM'] = macro['yield_10y'] - macro['yield_3m']
    macro['DEF'] = macro['yield_baa'] - macro['yield_aaa']
    macro = macro[['date', 'TERM', 'DEF']].dropna().reset_index(drop=True)

    macro['z_TERM'] = (macro['TERM'] - macro['TERM'].expanding().mean()) / macro['TERM'].expanding().std(ddof=1)
    macro['z_DEF'] = (macro['DEF'] - macro['DEF'].expanding().mean()) / macro['DEF'].expanding().std(ddof=1)
    macro['M_t'] = macro['z_TERM'] - macro['z_DEF']
    macro['expanding_median'] = macro['M_t'].expanding().median()

    macro['signal_ready'] = np.arange(len(macro)) >= 24
    macro['regime_label'] = np.where(
        macro['signal_ready'],
        np.where(macro['M_t'] > macro['expanding_median'], 'good', 'bad'),
        pd.NA,
    )
    macro['holding_date'] = macro['date'] + MonthEnd(1)
    return macro


macro_signal = build_macro_signal()
macro_signal[['date', 'TERM', 'DEF', 'M_t', 'expanding_median', 'regime_label', 'holding_date']].head(30)


## 9. Load Fama-French Five-Factor Data And Validate The Self-Built Factors

This section loads the Ken French five-factor file, converts all returns from percentages to decimals, and compares the self-built profitability and investment factors to `RMW` and `CMA` using simple summary diagnostics.


In [ ]:
def load_ff5(path: Path | None = None) -> pd.DataFrame:
    path = path or DATA_DIR / 'F-F_Research_Data_5_Factors_2x3_CSV' / 'F-F_Research_Data_5_Factors_2x3.csv'
    text = path.read_text(encoding='utf-8', errors='ignore').splitlines()
    header_idx = next(i for i, line in enumerate(text) if line.startswith(',Mkt-RF'))
    data_lines = [text[header_idx]]
    data_lines.extend(line for line in text[header_idx + 1:] if re.match(r'^\d{6},', line))

    ff5 = pd.read_csv(StringIO('
'.join(data_lines)))
    ff5 = ff5.rename(columns={ff5.columns[0]: 'yyyymm'})
    ff5['date'] = pd.to_datetime(ff5['yyyymm'].astype(str), format='%Y%m') + MonthEnd(0)

    factor_cols = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']
    for col in factor_cols:
        ff5[col] = pd.to_numeric(ff5[col], errors='coerce') / 100.0

    ff5 = ff5[['date'] + factor_cols].dropna().sort_values('date')
    ff5 = ff5.loc[ff5['date'].between(SAMPLE_START, SAMPLE_END)].reset_index(drop=True)
    return ff5


def build_factor_validation(factor_returns: pd.DataFrame, ff5: pd.DataFrame) -> pd.DataFrame:
    merged = factor_returns.merge(ff5[['date', 'RMW', 'CMA']], on='date', how='inner')
    diagnostics = []

    for series_name, benchmark_name in [('R_prof', 'RMW'), ('R_inv', 'CMA')]:
        sample = merged[[series_name, benchmark_name]].dropna()
        diagnostics.append(
            {
                'factor': series_name,
                'benchmark': benchmark_name,
                'correlation': sample[series_name].corr(sample[benchmark_name]),
                'mean': sample[series_name].mean(),
                'std': sample[series_name].std(ddof=1),
                'sharpe': sample[series_name].mean() / sample[series_name].std(ddof=1),
                'n_months': int(sample.shape[0]),
            }
        )

    return pd.DataFrame(diagnostics)


factor_returns = pd.concat([R_lowvol, R_prof, R_inv], axis=1).reset_index().rename(columns={'index': 'date'})
factor_returns = factor_returns.loc[factor_returns['date'].between(SAMPLE_START, SAMPLE_END)].copy()

ff5 = load_ff5()
factor_validation = build_factor_validation(factor_returns, ff5)
factor_validation


## 10. Combine Factor Returns With The Regime Signal

This section merges the live factor returns with the lagged regime classifications, applies the fixed good-state and bad-state weights from the strategy design, and computes the monthly strategy return series.


In [ ]:
def build_strategy_panel(factor_returns: pd.DataFrame, macro_signal: pd.DataFrame) -> pd.DataFrame:
    weights = macro_signal.loc[
        macro_signal['signal_ready'],
        ['holding_date', 'regime_label', 'M_t'],
    ].rename(columns={'holding_date': 'date'})

    panel = factor_returns.merge(weights, on='date', how='inner')
    panel = panel.dropna(subset=['R_lowvol', 'R_prof', 'R_inv', 'regime_label']).copy()

    panel['w_LV'] = np.where(panel['regime_label'] == 'good', GOOD_REGIME_WEIGHTS['w_LV'], BAD_REGIME_WEIGHTS['w_LV'])
    panel['w_P'] = np.where(panel['regime_label'] == 'good', GOOD_REGIME_WEIGHTS['w_P'], BAD_REGIME_WEIGHTS['w_P'])
    panel['w_I'] = np.where(panel['regime_label'] == 'good', GOOD_REGIME_WEIGHTS['w_I'], BAD_REGIME_WEIGHTS['w_I'])
    panel['R_strategy'] = (
        panel['w_LV'] * panel['R_lowvol']
        + panel['w_P'] * panel['R_prof']
        + panel['w_I'] * panel['R_inv']
    )

    panel = panel.loc[panel['date'].between(SAMPLE_START, SAMPLE_END)].sort_values('date').reset_index(drop=True)
    return panel


strategy_panel = build_strategy_panel(factor_returns=factor_returns, macro_signal=macro_signal)
strategy_panel[['date', 'regime_label', 'M_t', 'w_LV', 'w_P', 'w_I', 'R_lowvol', 'R_prof', 'R_inv', 'R_strategy']].head(12)


## 11. Validate And Export The Final Return Series

This section converts the strategy return series into the required two-column output, checks for missing values, infinite values, missing months, and implausible return magnitudes, and then writes the final CSV to disk.


In [ ]:
def validate_strategy_returns(strategy_returns: pd.DataFrame) -> None:
    if strategy_returns['ret'].isna().any():
        raise ValueError('Strategy return series contains NaN values.')
    if np.isinf(strategy_returns['ret']).any():
        raise ValueError('Strategy return series contains infinite values.')

    monthly_index = pd.period_range(
        strategy_returns['date'].min().to_period('M'),
        strategy_returns['date'].max().to_period('M'),
        freq='M',
    )
    actual_index = strategy_returns['date'].dt.to_period('M')
    if len(monthly_index) != actual_index.nunique():
        raise ValueError('Strategy return series has missing months.')
    if (strategy_returns['ret'].abs() > 0.5).any():
        raise ValueError('Strategy return series contains values outside the expected range [-0.5, 0.5].')


strategy_returns = strategy_panel[['date', 'R_strategy']].rename(columns={'R_strategy': 'ret'}).copy()
validate_strategy_returns(strategy_returns)

strategy_returns_for_export = strategy_returns.copy()
strategy_returns_for_export['date'] = strategy_returns_for_export['date'].dt.strftime('%Y-%m-%d')
strategy_returns_for_export.to_csv(OUTPUT_CSV, index=False)

strategy_returns_for_export.head()


## 12. Plot Monthly Strategy Returns By Regime

This section creates the chart that colors good-state monthly strategy returns in green and bad-state monthly strategy returns in red, and saves the figure as a PNG in the project folder.


In [ ]:
def plot_strategy_returns_by_regime(strategy_panel: pd.DataFrame, output_path: Path | None = None) -> Path:
    output_path = output_path or OUTPUT_PLOT
    plot_data = strategy_panel[['date', 'regime_label', 'R_strategy']].copy()
    colors = np.where(plot_data['regime_label'] == 'good', '#2e8b57', '#c0392b')

    fig, ax = plt.subplots(figsize=(16, 6))
    ax.bar(plot_data['date'], plot_data['R_strategy'], color=colors, width=25, linewidth=0)
    ax.axhline(0, color='black', linewidth=1, alpha=0.7)
    ax.set_title('Monthly Strategy Returns by Credit Regime')
    ax.set_xlabel('Date')
    ax.set_ylabel('Strategy Return')

    handles = [
        plt.Rectangle((0, 0), 1, 1, color='#2e8b57', label='Good regime'),
        plt.Rectangle((0, 0), 1, 1, color='#c0392b', label='Bad regime'),
    ]
    ax.legend(handles=handles, loc='upper right', frameon=False)
    fig.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches='tight')
    plt.close(fig)
    return output_path


plot_path = plot_strategy_returns_by_regime(strategy_panel)
plot_path


## 13. Summarize The Final Objects

This final section packages the core outputs into a single object and shows a few compact diagnostics so the notebook ends with the key strategy artifacts ready for the later performance-evaluation and robustness sections.


In [ ]:
artifacts = StrategyArtifacts(
    crsp=crsp,
    fundamentals=fundamentals,
    linked_signals=linked_signals,
    factor_returns=factor_returns,
    factor_validation=factor_validation,
    macro_signal=macro_signal,
    strategy_panel=strategy_panel,
    strategy_returns=strategy_returns_for_export,
)

print(f'Strategy output written to: {OUTPUT_CSV}')
print(f'Regime chart written to: {OUTPUT_PLOT}')
print(f'Monthly strategy returns: {len(strategy_returns_for_export):,}')
print(f"Date range: {strategy_returns_for_export['date'].iloc[0]} to {strategy_returns_for_export['date'].iloc[-1]}")

factor_validation


## 14. Build Benchmarks, Summary Tables, And Comparison Plot

This section builds the benchmark return series, computes the descriptive statistics, reshapes the main results into summary-ready tables, and produces the cumulative-return comparison chart for the strategy and benchmarks.


In [ ]:
from IPython.display import Image, display

from credit_factor_rotation_core import (
    OUTPUT_CUMULATIVE_PLOT,
    build_benchmarks,
    compute_turnover_from_weights,
    descriptive_statistics,
    load_ff5,
    load_qmj_returns,
    plot_cumulative_returns,
)

ff5_eval = load_ff5()
qmj_eval = load_qmj_returns()
benchmarks = build_benchmarks(strategy_panel, ff5_eval, qmj_eval)
turnover_series = compute_turnover_from_weights(strategy_panel[['date', 'w_LV', 'w_P', 'w_I']])
descriptive_stats_table = descriptive_statistics(
    benchmarks.rename(columns={'R_strategy': 'Strategy'}),
    turnover=turnover_series,
)

cumulative_plot_path = plot_cumulative_returns(
    benchmarks.rename(columns={'R_strategy': 'Strategy'}),
    OUTPUT_CUMULATIVE_PLOT,
    title='Cumulative Returns: Strategy vs Benchmarks',
)

descriptive_stats_table


## 15. Run CAPM, FF3, And FF5 Alpha Regressions

This section estimates the required asset-pricing regressions with Newey-West HAC standard errors using 6 lags, then reshapes the outputs into paper-ready summary and alpha tables. It also displays the cumulative comparison chart created in the previous step.


In [ ]:
from IPython.display import Image, display

from credit_factor_rotation_core import prepare_alpha_table, prepare_summary_table, run_factor_regressions

regression_results = run_factor_regressions(
    benchmarks.rename(columns={'R_strategy': 'Strategy'}),
    ff5_eval,
    nw_lags=6,
)
summary_table = prepare_summary_table(descriptive_stats_table, regression_results)
alpha_table = prepare_alpha_table(regression_results)

print('Summary table')
display(summary_table)
print('Alpha table')
display(alpha_table)
print('Cumulative comparison plot')
display(Image(filename=str(cumulative_plot_path)))


## 16. Run The Sub-Period Robustness Check

This section splits the sample at December 31, 2007 and recomputes descriptive statistics plus alpha outputs for the pre-crisis and post-crisis periods. This is the first robustness check from the strategy plan.


In [ ]:
from credit_factor_rotation_core import run_subperiod_analysis

subperiod_results = run_subperiod_analysis(
    benchmarks.rename(columns={'R_strategy': 'Strategy'}),
    ff5_eval,
    split_date='2007-12-31',
)

subperiod_results


## 17. Run The Alternative Regime Robustness Checks

This section evaluates the tercile threshold rule, the VIX-augmented signal, and the Fama-French `RMW+CMA` timing robustness, then compresses the robustness results into a single summary-ready table and displays a cumulative-return comparison plot across the main and robustness variants.


In [ ]:
from IPython.display import Image, display

from credit_factor_rotation_core import (
    OUTPUT_ROBUSTNESS_PLOT,
    build_robustness_summary,
    plot_cumulative_returns,
    run_ff_good_cash_strategy,
    run_ml_robustness,
    run_tercile_strategy,
    run_vix_strategy,
)

tercile_strategy = run_tercile_strategy(factor_returns)
vix_strategy = run_vix_strategy(factor_returns)
ff_good_cash_strategy = run_ff_good_cash_strategy(ff5=ff5_eval, macro_signal=macro_signal, sample_dates=strategy_returns['date'])
ml_strategy = run_ml_robustness(factor_returns)
robustness_summary = build_robustness_summary(
    main_strategy=strategy_returns.copy(),
    tercile_strategy=tercile_strategy,
    vix_strategy=vix_strategy,
    ff_good_cash_strategy=ff_good_cash_strategy,
    ml_strategy=ml_strategy,
    ff5=ff5_eval,
)

robustness_plot_frame = (
    strategy_returns.rename(columns={'ret': 'Main_Strategy'})
    .merge(tercile_strategy[['date', 'ret']].rename(columns={'ret': 'Tercile_Robustness'}), on='date', how='outer')
    .merge(vix_strategy[['date', 'ret']].rename(columns={'ret': 'VIX_Robustness'}), on='date', how='outer')
    .merge(ff_good_cash_strategy[['date', 'ret']].rename(columns={'ret': 'FF_Good_Cash_Robustness'}), on='date', how='outer')
    .merge(ml_strategy[['date', 'ret']].rename(columns={'ret': 'ML_Robustness'}), on='date', how='outer')
    .sort_values('date')
)
robustness_plot_path = plot_cumulative_returns(
    robustness_plot_frame,
    OUTPUT_ROBUSTNESS_PLOT,
    title='Cumulative Returns: Main Strategy vs Robustness Variants',
)

display(robustness_summary)
display(Image(filename=str(robustness_plot_path)))


## 18. Run The Fama-French Good-State Robustness Check

This section reuses the same credit-regime signal, but replaces the self-constructed factors with the Ken French `RMW` and `CMA` factors. The strategy earns `0.5 * (RMW + CMA)` in good regimes and sits in `RF` cash in bad regimes.


In [ ]:
from IPython.display import Image, display

from credit_factor_rotation_core import (
    OUTPUT_FF_GOOD_CASH_PLOT,
    descriptive_statistics,
    plot_cumulative_returns,
    prepare_summary_table,
    run_factor_regressions,
    run_ff_good_cash_strategy,
)

ff_good_cash_strategy = run_ff_good_cash_strategy(ff5=ff5_eval, macro_signal=macro_signal, sample_dates=strategy_returns['date'])
ff_good_cash_returns = ff_good_cash_strategy[['date', 'ret']].rename(columns={'ret': 'FF_Good_Cash_Robustness'})
ff_good_cash_stats = descriptive_statistics(ff_good_cash_returns)
ff_good_cash_regs = run_factor_regressions(ff_good_cash_returns, ff5_eval)
ff_good_cash_summary = prepare_summary_table(ff_good_cash_stats, ff_good_cash_regs)

ff_good_cash_plot_path = plot_cumulative_returns(
    strategy_returns.rename(columns={'ret': 'Main_Strategy'})
    .merge(ff_good_cash_returns, on='date', how='inner'),
    OUTPUT_FF_GOOD_CASH_PLOT,
    title='Cumulative Returns: Main Strategy vs FF RMW+CMA / Cash',
)

print('FF RMW+CMA / cash robustness summary')
display(ff_good_cash_summary)
print('FF RMW+CMA / cash sample')
display(ff_good_cash_strategy[['date', 'regime_label', 'risky_leg', 'RF', 'ret']].head())
print('FF RMW+CMA / cash comparison plot')
display(Image(filename=str(ff_good_cash_plot_path)))


## 19. Run The ML Robustness Check

This section runs the expanding-window gradient-boosted classifier, summarizes how often it agrees with the main classifier, reports the ML strategy performance diagnostics, and shows the average feature importances that drove the robustness results.


In [ ]:
from IPython.display import Image, display

from credit_factor_rotation_core import (
    OUTPUT_ML_IMPORTANCE_PLOT,
    plot_ml_feature_importance,
    summarize_ml_robustness,
)

ml_summary, ml_feature_importance = summarize_ml_robustness(
    ml_strategy=ml_strategy,
    main_strategy_panel=strategy_panel,
    ff5=ff5_eval,
)
ml_importance_plot_path = plot_ml_feature_importance(ml_feature_importance, OUTPUT_ML_IMPORTANCE_PLOT)

print('ML robustness summary')
display(ml_summary)
print('ML feature importance')
display(ml_feature_importance)
print('ML sample')
display(ml_strategy[['date', 'regime_label', 'prob_good', 'ret']].head())
print('ML feature importance plot')
display(Image(filename=str(ml_importance_plot_path)))


## 20. Export Final Tables And Plots

This final section exports the summary-ready tables and plots to disk so the paper write-up can use stable files instead of rerunning ad hoc notebook cells.


In [ ]:
from credit_factor_rotation_core import export_report_outputs, run_full_strategy

full_artifacts = run_full_strategy(write_csv=True)
report_paths = export_report_outputs(full_artifacts)

for name, report_path in report_paths.items():
    print(f'{name}: {report_path}')
